In [1]:
import os
import pickle
import pandas as pd

In [2]:
# Define the directory containing the CSV files
shap_directory = "/home/guillen/Conference Paper SHAP Maciej actualizado/Satimage/shap"

parquet_directory = "/home/guillen/Conference Paper SHAP Maciej actualizado/Satimage/shap"

In [3]:
featuresList = ['Aattr', 'Battr', 'Cattr', 'Dattr', 'Eattr', 'Fattr', 'A1attr',
       'B2attr', 'C3attr', 'D4attr', 'E5attr', 'F6attr', 'A7attr', 'B8attr',
       'C9attr', 'D10attr', 'E11attr', 'F12attr', 'A13attr', 'B14attr',
       'C15attr', 'D16attr', 'E17attr', 'F18attr', 'A19attr', 'B20attr',
       'C21attr', 'D22attr', 'E23attr', 'F24attr', 'A25attr', 'B26attr',
       'C27attr', 'D28attr', 'E29attr', 'F30attr']

#read .csv files
# Get all files starting with "shap_list_ir" and ending with ".csv"
csv_files = [f for f in os.listdir(shap_directory) if f.startswith("shap_list_ir") and f.endswith(".csv")]

%%time
#read parquet files

# Get all files starting with "shap_list_ir" and ending with ".parquet"
parquet_files = [f for f in os.listdir(shap_directory) if f.startswith("shap_list_ir") and f.endswith(".parquet")]

# Iterate through parquet files and process them
for file in parquet_files:
    file_path = os.path.join(shap_directory, file)
    # Read the parquet file into a DataFrame
    df = pd.read_parquet(file_path)
    # Process the DataFrame (example: print first 5 rows)
    print(f"Processing file: {file}")
    #print(df.head())

%%time
# Save as parquet

# Loop through the files and save each as a Parquet file
for csv_file in csv_files:
    # Full path to the CSV file
    csv_path = os.path.join(shap_directory, csv_file)
    
    # Read the CSV file into a DataFrame
    df = pd.read_csv(csv_path)
    
    # Construct the output Parquet file path
    parquet_file = os.path.join(parquet_directory, f"{os.path.splitext(csv_file)[0]}.parquet")
    
    # Save the DataFrame as a Parquet file
    df.to_parquet(parquet_file, index=False)
    

# .pkl files version

# Get all files in the directory starting with "shap_list_ir" and ending with ".pkl"
pkl_files = [f for f in os.listdir(shap_directory) if f.startswith("shap_list_ir") and f.endswith(".pkl")]

# Load each file into a variable with the same name as the file
for pkl_file in pkl_files:
    variable_name = pkl_file.replace(".pkl", "")  # Remove .pkl to create the variable name
    file_path = os.path.join(shap_directory, pkl_file)  # Full path to the file
    
    # Load the .pkl file
    with open(file_path, 'rb') as f:
        globals()[variable_name] = pickle.load(f)


## Data preparation

### Selection of Shap values for true class

In [4]:
%%time
# Dictionary to store DataFrames temporarily by their base name (excluding class label)
dataframes = {}

# Get all files starting with "shap_list_ir" and ending with ".parquet"
parquet_files = [f for f in os.listdir(shap_directory) if f.startswith("mlp_shap_list_ir") and f.endswith(".parquet")]

# Load each file into a DataFrame, add 'shapClass' column
for file in parquet_files:
    # Determine the class value from the filename
    class_value = 0 if "_class0" in file else 1  # Extract class from filename
    
    # Base name without the class label and extension
    base_name = file.replace("_class0.parquet", "").replace("_class1.parquet", "")
    
    # Load the CSV file into a DataFrame and add 'shapClass' column
    file_path = os.path.join(shap_directory, file)
    df = pd.read_parquet(file_path)
    df['shapClass'] = class_value
    
    # Store the DataFrame in a dictionary under its base name
    if base_name not in dataframes:
        dataframes[base_name] = []
    dataframes[base_name].append(df)



# Define a list to store the names of the new DataFrames
subsetList = []

# Combine paired DataFrames and create new variables
for base_name, df_list in dataframes.items():
    if len(df_list) == 2:  # Ensure we have both class0 and class1 DataFrames
        combined_df = pd.concat(df_list, ignore_index=True)  # Combine the two DataFrames
        
        # Create a variable in the global namespace with the base_name
        globals()[base_name] = combined_df
        
        # Add the base_name to the subsetList
        subsetList.append(base_name)

# Print the list of new DataFrames
#print(f"Combined DataFrames created: {', '.join(subsetList)}")


CPU times: user 22.3 s, sys: 8.06 s, total: 30.4 s
Wall time: 4min 38s


In [5]:
# Replace values in Real_class and Predicted_class for each DataFrame in subsetList
for df_name in subsetList:
    df = globals()[df_name]  # Access the DataFrame by its name
    df['Real_class'] = df['Real_class'].replace({'NO': 0, 'SI': 1})
    df['Predicted_class'] = df['Predicted_class'].replace({'NO': 0, 'SI': 1})

/tmp/ipykernel_1532041/1657802143.py:4: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['Real_class'] = df['Real_class'].replace({'NO': 0, 'SI': 1})
/tmp/ipykernel_1532041/1657802143.py:5: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['Predicted_class'] = df['Predicted_class'].replace({'NO': 0, 'SI': 1})


In [6]:
# Filter rows where Predicted_class equals shapClass for each DataFrame in subsetList
for df_name in subsetList:
    df = globals()[df_name]  # Access the DataFrame by its name
    
    # Filter rows with matching Predicted_class and shapClass
    df_filtered = df.loc[df['Predicted_class'] == df['shapClass']]
       
    # Overwrite the original DataFrame with the filtered DataFrame
    globals()[df_name] = df_filtered

In [7]:
len(subsetList)

510

In [8]:
mlp_shap_list_ir_1_subset_10.columns

Index(['Aattr', 'Battr', 'Cattr', 'Dattr', 'Eattr', 'Fattr', 'A1attr',
       'B2attr', 'C3attr', 'D4attr', 'E5attr', 'F6attr', 'A7attr', 'B8attr',
       'C9attr', 'D10attr', 'E11attr', 'F12attr', 'A13attr', 'B14attr',
       'C15attr', 'D16attr', 'E17attr', 'F18attr', 'A19attr', 'B20attr',
       'C21attr', 'D22attr', 'E23attr', 'F24attr', 'A25attr', 'B26attr',
       'C27attr', 'D28attr', 'E29attr', 'F30attr', 'PATIENT_ID', 'Real_class',
       'Predicted_class', 'Model_Output', 'shapClass'],
      dtype='object')

# Scoring calculation

## Order indicator

In [9]:
df_filtered.columns

Index(['Aattr', 'Battr', 'Cattr', 'Dattr', 'Eattr', 'Fattr', 'A1attr',
       'B2attr', 'C3attr', 'D4attr', 'E5attr', 'F6attr', 'A7attr', 'B8attr',
       'C9attr', 'D10attr', 'E11attr', 'F12attr', 'A13attr', 'B14attr',
       'C15attr', 'D16attr', 'E17attr', 'F18attr', 'A19attr', 'B20attr',
       'C21attr', 'D22attr', 'E23attr', 'F24attr', 'A25attr', 'B26attr',
       'C27attr', 'D28attr', 'E29attr', 'F30attr', 'PATIENT_ID', 'Real_class',
       'Predicted_class', 'Model_Output', 'shapClass'],
      dtype='object')

In [10]:
%%time
# Define sorting preferences for each indicator
sort_order_mean = 'descending'    # Options: 'ascending' or 'descending'
sort_order_std = 'descending'
sort_order_sum = 'descending'

# Define the list of features (replace with actual feature names)
#featuresList = ['feature_0', 'feature_1', 'feature_2', ...]  # Add all your feature names here

# Dictionary to store sorted feature lists for each indicator
sorted_features = {}
sorted_values = {}
# Loop through each DataFrame in subsetList
for df_name in subsetList:
    df = globals()[df_name]  # Access the DataFrame by its name
    
    # Calculate absolute values of the features
    df_abs = df[featuresList].abs()
    
    # Calculate mean, std, and sum for each feature
    feature_mean = df_abs.mean()
    feature_std = df_abs.std()
    feature_sum = df_abs.sum()
    
    # Sort feature names based on each indicator
    sorted_mean = feature_mean.sort_values(ascending=(sort_order_mean == 'ascending')).index.tolist()
    sorted_std = feature_std.sort_values(ascending=(sort_order_std == 'ascending')).index.tolist()
    sorted_sum = feature_sum.sort_values(ascending=(sort_order_sum == 'ascending')).index.tolist()
    
    # Store the sorted lists in the dictionary
    sorted_features[df_name] = {
        'orderMean': sorted_mean,
        'orderStd': sorted_std,
        'orderSum': sorted_sum}
        
        # Store the sorted lists in the dictionary
    sorted_values[df_name] = {
        'shapMean': feature_mean,
        'shapStd': feature_std,
        'shapSum': feature_sum
    }

# # Print results
# for df_name, orders in sorted_features.items():
#     print(f"DataFrame: {df_name}")
#     print(f"  Features sorted by Mean: {orders['orderMean']}")
#     print(f"  Features sorted by Std:  {orders['orderStd']}")
#     print(f"  Features sorted by Sum:  {orders['orderSum']}")
#     print()  # Add spacing between DataFrames


CPU times: user 4.6 s, sys: 10 ms, total: 4.61 s
Wall time: 54.5 s


In [11]:
%%time
# Initialize an empty list to collect DataFrames
dfs = []

# Loop through the subsets in the dictionary
for key, value in sorted_values.items():
    
    # Convert the nested dictionary to a DataFrame
    df = pd.DataFrame(value)
    # Add the subset identifier as a new column
    df['subset'] = key
    
    imbalance_ratio = "_".join(key.split('_')[:-2])
    df['imbalance_ratio'] = imbalance_ratio
    # Append the DataFrame to the list
    dfs.append(df)

# Concatenate all DataFrames from the list into a single DataFrame
sorted_orderValuesScore = pd.concat(dfs, axis=0)

# Reset index to make it clean
sorted_orderValuesScore.reset_index(inplace=True)

CPU times: user 495 ms, sys: 0 ns, total: 495 ms
Wall time: 8.85 s


In [12]:
sorted_orderValuesScore

,index,shapMean,shapStd,shapSum,subset,imbalance_ratio
0,Aattr,0.001789,0.005018,4.472011,mlp_shap_list_ir_1_subset_4,mlp_shap_list_ir_1
1,Battr,0.003941,0.009097,9.852009,mlp_shap_list_ir_1_subset_4,mlp_shap_list_ir_1
2,Cattr,0.001229,0.003563,3.072923,mlp_shap_list_ir_1_subset_4,mlp_shap_list_ir_1
3,Dattr,0.002095,0.005767,5.237618,mlp_shap_list_ir_1_subset_4,mlp_shap_list_ir_1
4,Eattr,0.005846,0.015631,14.615618,mlp_shap_list_ir_1_subset_4,mlp_shap_list_ir_1
...,...,...,...,...,...,...
18355,B26attr,0.004820,0.011485,12.048853,mlp_shap_list_ir_2_subset_8,mlp_shap_list_ir_2
18356,C27attr,0.007467,0.015552,18.668196,mlp_shap_list_ir_2_subset_8,mlp_shap_list_ir_2
18357,D28attr,0.008599,0.019774,21.497337,mlp_shap_list_ir_2_subset_8,mlp_shap_list_ir_2
18358,E29attr,0.007019,0.015243,17.546259,mlp_shap_list_ir_2_subset_8,mlp_shap_list_ir_2


In [13]:
%%time

# Flatten the sorted_features dictionary into rows for a DataFrame
rows = []
for df_name, orders in sorted_features.items():
    for indicator, sorted_list in orders.items():
        # Extract the imbalance ratio (prefix before "_subset")
        imbalance_ratio = "_".join(df_name.split('_')[:-2]) 
        
        
        rows.append({
            'imbalanceRatio': imbalance_ratio,
            'Subset': df_name,
            'Indicator': indicator,
            'Sorted_Features': sorted_list
        })

# Convert the list of rows into a DataFrame
sorted_orderScore = pd.DataFrame(rows)

CPU times: user 11.7 ms, sys: 2.89 ms, total: 14.6 ms
Wall time: 399 ms


In [14]:
sorted_orderScore

,imbalanceRatio,Subset,Indicator,Sorted_Features
0,mlp_shap_list_ir_1,mlp_shap_list_ir_1_subset_4,orderMean,"[F24attr, F18attr, Eattr, A7attr, D16attr, E17..."
1,mlp_shap_list_ir_1,mlp_shap_list_ir_1_subset_4,orderStd,"[F24attr, F18attr, Eattr, A7attr, E17attr, B20..."
2,mlp_shap_list_ir_1,mlp_shap_list_ir_1_subset_4,orderSum,"[F24attr, F18attr, Eattr, A7attr, D16attr, E17..."
3,mlp_shap_list_ir_2,mlp_shap_list_ir_2_subset_14,orderMean,"[A7attr, C15attr, D10attr, Battr, C3attr, A19a..."
4,mlp_shap_list_ir_2,mlp_shap_list_ir_2_subset_14,orderStd,"[C3attr, Battr, D4attr, A19attr, A7attr, E11at..."
...,...,...,...,...
1525,mlp_shap_list_ir_3,mlp_shap_list_ir_3_subset_66,orderStd,"[E5attr, F12attr, C15attr, B8attr, F18attr, A1..."
1526,mlp_shap_list_ir_3,mlp_shap_list_ir_3_subset_66,orderSum,"[D16attr, F12attr, E11attr, F18attr, B20attr, ..."
1527,mlp_shap_list_ir_2,mlp_shap_list_ir_2_subset_8,orderMean,"[B8attr, F12attr, Fattr, D16attr, A13attr, F24..."
1528,mlp_shap_list_ir_2,mlp_shap_list_ir_2_subset_8,orderStd,"[B8attr, Fattr, E23attr, A25attr, F24attr, A13..."


In [15]:
sorted_orderValuesScore

,index,shapMean,shapStd,shapSum,subset,imbalance_ratio
0,Aattr,0.001789,0.005018,4.472011,mlp_shap_list_ir_1_subset_4,mlp_shap_list_ir_1
1,Battr,0.003941,0.009097,9.852009,mlp_shap_list_ir_1_subset_4,mlp_shap_list_ir_1
2,Cattr,0.001229,0.003563,3.072923,mlp_shap_list_ir_1_subset_4,mlp_shap_list_ir_1
3,Dattr,0.002095,0.005767,5.237618,mlp_shap_list_ir_1_subset_4,mlp_shap_list_ir_1
4,Eattr,0.005846,0.015631,14.615618,mlp_shap_list_ir_1_subset_4,mlp_shap_list_ir_1
...,...,...,...,...,...,...
18355,B26attr,0.004820,0.011485,12.048853,mlp_shap_list_ir_2_subset_8,mlp_shap_list_ir_2
18356,C27attr,0.007467,0.015552,18.668196,mlp_shap_list_ir_2_subset_8,mlp_shap_list_ir_2
18357,D28attr,0.008599,0.019774,21.497337,mlp_shap_list_ir_2_subset_8,mlp_shap_list_ir_2
18358,E29attr,0.007019,0.015243,17.546259,mlp_shap_list_ir_2_subset_8,mlp_shap_list_ir_2


In [16]:
sorted_orderValuesScore.to_csv("sorted_orderValuesScore_mlp.csv")

In [17]:
sorted_orderScore.to_csv("shap_orderScore_mlp.csv")

***
***
# in progress/trash

## Distance indicator

%%time
import matplotlib.pyplot as plt

# Assuming 'shap_list_ir_1_subset_1' and 'featuresList' are already defined
df = shap_list_ir_1_subset_248  # Access the DataFrame

# Create histograms for each feature
for feature in featuresList:
    plt.figure(figsize=(6, 4))  # Set figure size
    plt.hist(df[feature], bins=20, color='skyblue', edgecolor='black')
    plt.title(f'Histogram for {feature}')
    plt.xlabel('Value')
    plt.ylabel('Frequency')
    plt.grid(True)
    plt.show()

    
    # Save the histogram as a PNG file in the specified folder
    filename = os.path.join(output_folder, f"Histogram_for_{feature.replace(' ', '_')}.png")  # Replace spaces with underscores for filenames
    plt.savefig(filename, dpi=300, bbox_inches='tight')

## max density

In [18]:
%%time
import numpy as np
import pandas as pd

# List to store the results
histogram_results = []

# Loop through each DataFrame in subsetList
for df_name in subsetList:
    df = globals()[df_name]  # Access the DataFrame by its name
    
    # Loop through each feature in featuresList
    for feature in featuresList:
        # Calculate histogram data (20 bins, default behavior)
        values, bins = np.histogram(df[feature].dropna(), bins=20)
        
        # Sort frequencies to get the top 3 points
        top_indices = np.argsort(values)[-3:][::-1]  # Get indices of 3 highest frequencies in descending order
        
        # Calculate the values corresponding to the highest frequencies (bin centers)
        bin_centers = (bins[:-1] + bins[1:]) / 2  # Calculate bin centers
        top_frequencies = [bin_centers[i] for i in top_indices]
        
        # Append results: subset name and top 3 frequency points
        histogram_results.append({
            'Subset': f'{df_name}_{feature}',
            'freq1': top_frequencies[0],
            'freq2': top_frequencies[1],
            'freq3': top_frequencies[2]
        })

# Create a DataFrame from the results
maxDensityValues = pd.DataFrame(histogram_results)

# Display the resulting DataFrame
print(maxDensityValues)


                                    Subset     freq1     freq2     freq3
0        mlp_shap_list_ir_1_subset_4_Aattr  0.002009 -0.001868  0.005886
1        mlp_shap_list_ir_1_subset_4_Battr  0.003989 -0.006484 -0.016957
2        mlp_shap_list_ir_1_subset_4_Cattr -0.002189  0.002826  0.007842
3        mlp_shap_list_ir_1_subset_4_Dattr -0.005342  0.007788 -0.018472
4        mlp_shap_list_ir_1_subset_4_Eattr  0.002469  0.037651 -0.032712
...                                    ...       ...       ...       ...
18355  mlp_shap_list_ir_2_subset_8_B26attr  0.003813 -0.017062  0.024689
18356  mlp_shap_list_ir_2_subset_8_C27attr  0.002626  0.028430 -0.023177
18357  mlp_shap_list_ir_2_subset_8_D28attr -0.025210  0.030717  0.086645
18358  mlp_shap_list_ir_2_subset_8_E29attr -0.002063  0.026623 -0.030748
18359  mlp_shap_list_ir_2_subset_8_F30attr  0.000833 -0.009713  0.011378

[18360 rows x 4 columns]
CPU times: user 13.1 s, sys: 133 ms, total: 13.2 s
Wall time: 2min 3s


In [19]:
%%time
import numpy as np
import pandas as pd

# Define the number of bins as a variable
num_bins = 20  # You can change this value as needed

# List to store the results
histogram_results = []

# Loop through each DataFrame in subsetList
for df_name in subsetList:
    df = globals()[df_name]  # Access the DataFrame by its name
    
    # Loop through each feature in featuresList
    for feature in featuresList:
        # Calculate histogram data
        values, bins = np.histogram(df[feature].dropna(), bins=num_bins)
        
        # Sort frequencies to get the top 3 points
        top_indices = np.argsort(values)[-3:][::-1]  # Get indices of 3 highest frequencies in descending order
        
        # Calculate the values corresponding to the highest frequencies (bin centers)
        bin_centers = (bins[:-1] + bins[1:]) / 2  # Calculate bin centers
        top_frequencies = [bin_centers[i] for i in top_indices]
        
        # Extract the imbalance ratio (prefix before "_subset")
        imbalance_ratio = "_".join(df_name.split('_')[:-2]) 
        
        # Append results: subset name, top 3 frequency points, imbalance ratio, and feature name
        histogram_results.append({
            'imbalanceRatio': imbalance_ratio,
            'Subset': df_name,
            'feature': feature, 
            'Indicator': f'localMax_{feature}',             
            'freq1': top_frequencies[0],
            'freq2': top_frequencies[1],
            'freq3': top_frequencies[2]         
        })

# Create a DataFrame from the results
maxDensityValues = pd.DataFrame(histogram_results)

print("Wrong calculation of freq2 and freq3, ther are next 2 maxes, not next 2 density points, freq1 is ok")

Wrong calculation of freq2 and freq3, ther are next 2 maxes, not next 2 density points, freq1 is ok
CPU times: user 10.7 s, sys: 30.5 ms, total: 10.7 s
Wall time: 1min 42s


In [20]:
maxDensityValues

,imbalanceRatio,Subset,feature,Indicator,freq1,freq2,freq3
0,mlp_shap_list_ir_1,mlp_shap_list_ir_1_subset_4,Aattr,localMax_Aattr,0.002009,-0.001868,0.005886
1,mlp_shap_list_ir_1,mlp_shap_list_ir_1_subset_4,Battr,localMax_Battr,0.003989,-0.006484,-0.016957
2,mlp_shap_list_ir_1,mlp_shap_list_ir_1_subset_4,Cattr,localMax_Cattr,-0.002189,0.002826,0.007842
3,mlp_shap_list_ir_1,mlp_shap_list_ir_1_subset_4,Dattr,localMax_Dattr,-0.005342,0.007788,-0.018472
4,mlp_shap_list_ir_1,mlp_shap_list_ir_1_subset_4,Eattr,localMax_Eattr,0.002469,0.037651,-0.032712
...,...,...,...,...,...,...,...
18355,mlp_shap_list_ir_2,mlp_shap_list_ir_2_subset_8,B26attr,localMax_B26attr,0.003813,-0.017062,0.024689
18356,mlp_shap_list_ir_2,mlp_shap_list_ir_2_subset_8,C27attr,localMax_C27attr,0.002626,0.028430,-0.023177
18357,mlp_shap_list_ir_2,mlp_shap_list_ir_2_subset_8,D28attr,localMax_D28attr,-0.025210,0.030717,0.086645
18358,mlp_shap_list_ir_2,mlp_shap_list_ir_2_subset_8,E29attr,localMax_E29attr,-0.002063,0.026623,-0.030748


In [21]:
%%time
# Dictionary for calculation the distance between the most important features from orderScore

# Assuming df is your DataFrame
localMaxDict = {}

# Create the localMaxDict using Subset as the key and {'feature': feature, 'freq1': freq1} as the value
localMaxDict = {
    row['Subset']: {'feature': row['feature'], 'freq1': row['freq1']}
    for index, row in maxDensityValues.iterrows()
}
    

# # Display the dictionary
# print(localMaxDict)


CPU times: user 1.46 s, sys: 8.85 ms, total: 1.47 s
Wall time: 7.77 s


In [22]:
maxDensityValues.to_csv("shap_maxDensityValues_mlp.csv")

In [23]:
maxDensityValues.columns

Index(['imbalanceRatio', 'Subset', 'feature', 'Indicator', 'freq1', 'freq2',
       'freq3'],
      dtype='object')

In [24]:
# Perform the merge operation on the "Subset" column
distanceScore_df = pd.merge(sorted_orderScore, maxDensityValues, on="Subset", how="inner")


In [25]:
maxDensityValues

,imbalanceRatio,Subset,feature,Indicator,freq1,freq2,freq3
0,mlp_shap_list_ir_1,mlp_shap_list_ir_1_subset_4,Aattr,localMax_Aattr,0.002009,-0.001868,0.005886
1,mlp_shap_list_ir_1,mlp_shap_list_ir_1_subset_4,Battr,localMax_Battr,0.003989,-0.006484,-0.016957
2,mlp_shap_list_ir_1,mlp_shap_list_ir_1_subset_4,Cattr,localMax_Cattr,-0.002189,0.002826,0.007842
3,mlp_shap_list_ir_1,mlp_shap_list_ir_1_subset_4,Dattr,localMax_Dattr,-0.005342,0.007788,-0.018472
4,mlp_shap_list_ir_1,mlp_shap_list_ir_1_subset_4,Eattr,localMax_Eattr,0.002469,0.037651,-0.032712
...,...,...,...,...,...,...,...
18355,mlp_shap_list_ir_2,mlp_shap_list_ir_2_subset_8,B26attr,localMax_B26attr,0.003813,-0.017062,0.024689
18356,mlp_shap_list_ir_2,mlp_shap_list_ir_2_subset_8,C27attr,localMax_C27attr,0.002626,0.028430,-0.023177
18357,mlp_shap_list_ir_2,mlp_shap_list_ir_2_subset_8,D28attr,localMax_D28attr,-0.025210,0.030717,0.086645
18358,mlp_shap_list_ir_2,mlp_shap_list_ir_2_subset_8,E29attr,localMax_E29attr,-0.002063,0.026623,-0.030748


In [26]:
sorted_orderScore

,imbalanceRatio,Subset,Indicator,Sorted_Features
0,mlp_shap_list_ir_1,mlp_shap_list_ir_1_subset_4,orderMean,"[F24attr, F18attr, Eattr, A7attr, D16attr, E17..."
1,mlp_shap_list_ir_1,mlp_shap_list_ir_1_subset_4,orderStd,"[F24attr, F18attr, Eattr, A7attr, E17attr, B20..."
2,mlp_shap_list_ir_1,mlp_shap_list_ir_1_subset_4,orderSum,"[F24attr, F18attr, Eattr, A7attr, D16attr, E17..."
3,mlp_shap_list_ir_2,mlp_shap_list_ir_2_subset_14,orderMean,"[A7attr, C15attr, D10attr, Battr, C3attr, A19a..."
4,mlp_shap_list_ir_2,mlp_shap_list_ir_2_subset_14,orderStd,"[C3attr, Battr, D4attr, A19attr, A7attr, E11at..."
...,...,...,...,...
1525,mlp_shap_list_ir_3,mlp_shap_list_ir_3_subset_66,orderStd,"[E5attr, F12attr, C15attr, B8attr, F18attr, A1..."
1526,mlp_shap_list_ir_3,mlp_shap_list_ir_3_subset_66,orderSum,"[D16attr, F12attr, E11attr, F18attr, B20attr, ..."
1527,mlp_shap_list_ir_2,mlp_shap_list_ir_2_subset_8,orderMean,"[B8attr, F12attr, Fattr, D16attr, A13attr, F24..."
1528,mlp_shap_list_ir_2,mlp_shap_list_ir_2_subset_8,orderStd,"[B8attr, Fattr, E23attr, A25attr, F24attr, A13..."


In [ ]:
%%time
# Function to replace feature names with corresponding freq1 values
def replace_feature_with_freq(sorted_features, subset, localMaxDict):
    updated_features = []
    
    # Iterate through each feature in the Sorted_Features list
    for feature in sorted_features:
        # Look up the freq1 value in localMaxDict using the (Subset, feature) key
        key = (subset, feature)
        
        # If the key exists in the localMaxDict, append the freq1 value
        if key in localMaxDict:
            updated_features.append(localMaxDict[key]['freq1'])
        else:
            updated_features.append(None)  # If no match, append None or handle accordingly
    
    return updated_features


In [ ]:

# Apply the function to replace feature names with freq1 values
sorted_orderScore['maxLocalOrder'] = sorted_orderScore.apply(lambda row: replace_feature_with_freq(row['Sorted_Features'], row['Subset'], localMaxDict), axis=1)


In [ ]:
sorted_orderScore

# Assuming 'shap_list_ir_1_subset_1' and 'featuresList' are already defined
df = maxDensityValues['feature']  # Access the DataFrame

# Define the folder to save histograms
output_folder = "maxDensity-histograms"
os.makedirs(output_folder, exist_ok=True)  # Create the folder if it doesn't exist

    # Create histograms for each feature
for feature in featuresList:
    plt.figure(figsize=(6, 4))  # Set figure size
    plt.hist(maxDensityValues[maxDensityValues['feature'] == feature]['freq1'], bins=20, color='skyblue', edgecolor='black')
    plt.title(f'Histogram for {feature}')
    plt.xlabel('Value')
    plt.ylabel('Frequency')
    plt.grid(True)
    
    # Save the histogram as a PNG file in the specified folder
    filename = os.path.join(output_folder, f"Histogram_for_{feature.replace(' ', '_')}.png")  # Replace spaces with underscores for filenames
    plt.savefig(filename, dpi=300, bbox_inches='tight')

***
***
## TEMP

In [ ]:
# Function to replace feature names with corresponding freq1 values
def replace_features_with_freq(row, localMaxDict):
    subset_name = row['Subset']
    sorted_features = row['Sorted_Features']
    
    updated_sorted_features = []
    
    # Iterate through the feature names in the Sorted_Features list
    for feature in sorted_features:
        # Create a key to check in localMaxDict (using the subset and feature)
        key = (subset_name, feature)
        
        # Look up the freq1 value in localMaxDict based on the key
        if key in localMaxDict:
            updated_sorted_features.append(localMaxDict[key]['freq1'])  # Append the freq1 value
        else:
            updated_sorted_features.append(None)  # If no match is found, add None (or handle as needed)
    
    return updated_sorted_features

# Apply the function to the dataframe
sorted_orderScore['maxLocalOrder'] = sorted_orderScore.apply(replace_features_with_freq, axis=1, localMaxDict=localMaxDict)

# Display the updated dataframe with the new 'maxLocalOrder' column
print(sorted_orderScore[['Subset', 'Indicator', 'Sorted_Features', 'maxLocalOrder']])


In [ ]:
aaa = sorted_orderScore.apply(replace_feature_with_freq, axis=1, localMaxDict=localMaxDict)

In [ ]:
aaa

In [ ]:
# Function to replace feature names with corresponding freq1 values from localMaxDict
def replace_feature_with_freq(row, localMaxDict):
    subset_name = row['Subset']
    indicator = row['Indicator']
    sorted_features = row['Sorted_Features']
    
    # Create the key based on Subset and Indicator
    key = (subset_name, f'localMax_{indicator}')  # Use 'localMax_' prefix to match the keys in localMaxDict
    
    # Initialize an empty list for storing the replaced values
    updated_sorted_features = []
    
    # Check if the combination of Subset and Indicator exists in localMaxDict
    if key in localMaxDict:
        # Get the mapping of features to freq1 for this key
        feature_to_freq1 = localMaxDict[key]
        
        # Iterate through the sorted_features list and replace feature with freq1 value if it matches
        for feature in sorted_features:
            # If feature matches the one in the dictionary, replace it with freq1
            if feature == feature_to_freq1['feature']:
                updated_sorted_features.append(feature_to_freq1['freq1'])
            else:
                updated_sorted_features.append(feature)
    else:
        # If key doesn't exist, just use the original Sorted_Features list
        updated_sorted_features = sorted_features
    
    return updated_sorted_features

# # Apply the function to the dataframe
# sorted_orderScore['localMaxOrder'] = sorted_orderScore.apply(replace_feature_with_freq, axis=1, localMaxDict=localMaxDict)

# # Display the updated dataframe
# print(sorted_orderScore[['Subset', 'Indicator', 'Sorted_Features', 'localMaxOrder']])


In [ ]:
aaa = sorted_orderScore.apply(replace_feature_with_freq, axis=1, localMaxDict=localMaxDict)

In [ ]:
aaa

In [ ]:
aaa[0]

In [ ]:
# Function to replace feature names with corresponding freq1 values
def replace_features_with_freq(row):
    # Replace the feature names in "Sorted_Features" with the "freq1" value
    sorted_features = row['Sorted_Features']
    feature_list = row['feature']
    freq1 = row['freq1']
    
    # Replace feature names with corresponding freq1 value
    updated_sorted_features = [freq1 if feature == feature_name else feature_name 
                               for feature, feature_name in zip(sorted_features, feature_list)]
    
    return updated_sorted_features

# # Apply the function to the merged dataframe
# merged_df['Sorted_Features'] = merged_df.apply(replace_features_with_freq, axis=1)

# # Display the updated DataFrame
# print(merged_df)


In [ ]:
# Function to replace the feature in Sorted_Features with corresponding freq1 value
def replace_feature_with_freq(row):
    sorted_features = row['Sorted_Features']  # List of features
    feature_name = row['feature']  # The feature to match
    freq1_value = row['freq1']  # The corresponding freq1 value
    
    # Replace the feature name in Sorted_Features with freq1_value if it matches the feature_name
    updated_sorted_features = [
        freq1_value if feature == feature_name else feature
        for feature in sorted_features
    ]
    
    return updated_sorted_features

# Apply the function to the merged dataframe
merged_df['Sorted_Features'] = merged_df.apply(replace_feature_with_freq, axis=1)

# Display the updated DataFrame
print(merged_df)


In [ ]:
distanceScore_df['ind_Features'] = distanceScore_df.apply(replace_features_with_freq, axis=1)

In [ ]:
distDf = distanceScore_df.copy()
distDf['localMaxOrder'] = distanceScore_df.apply(replace_features_with_freq, axis=1)

In [ ]:
distanceScore_df

In [ ]:
distDf